# Ariketa 3.2 — KNN eta erregresio logistikoa

**5072 · 3. DU** · Titanic (Colab)

Aurreko saioetan:

- **Decision Tree** → galdera/arau bidez sailkatzen ikusi dugu.

Saio honetan:

- **KNN** → antzekotasunaren bidez sailkatuko dugu.
- **Erregresio logistikoa** → probabilitatea erabiliko dugu.

### Ideia nagusiak

> **Decision Tree:** “Zein galdera bereizten ditu klaseak?”

> **KNN:** “Zein training kasu dira puntu berriaren antzekoenak?”

> **Logistic Regression:** “Zein probabilitate du kasuak 1 klasekoa izateko?”


## 📝 Nola egin notebook hau — 3 modu

- 🟢 **KOMENTATU:** kodea emanda dago; `#`-aren ondoren azaldu lerro bakoitzak zer egiten duen.
- 🟡 **HUTSUNEAK BETE:** `____` dauden tokiak osatu.
- 🔴 **ARIKETA GEHIGARRIA:** kodea zerotik idatzi edo esperimentua zabaldu.


# 0. Prestaketa 🟢 — datu-ihesa saihestuz

Aurreko bertsioetan ohikoa da `age` mediana eta `embarked` moda **dataset osoarekin** kalkulatzea.

Hemen praktika zuzenagoa erabiliko dugu:

1. lehenengo train / test banaketa;
2. mediana eta moda **TRAIN multzoarekin bakarrik** kalkulatu;
3. balio horiek train eta test multzoetan aplikatu;
4. kategorikoak kodetu;
5. aldagai numerikoak bakarrik eskalatu.

> **Arau nagusia:** datuetatik zerbait “ikasi” behar duen urratsa → TRAIN-ekin bakarrik.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

#
df = sns.load_dataset('titanic')

#
d = df[
    ['survived', 'pclass', 'sex', 'age', 'fare', 'embarked']
].copy()

#
train_df, test_df = train_test_split(
    d,
    test_size=0.2,
    random_state=42,
    stratify=d['survived']
)

#
age_mediana = train_df['age'].median()
embarked_moda = train_df['embarked'].mode()[0]

#
train_df['age'] = train_df['age'].fillna(age_mediana)
train_df['embarked'] = train_df['embarked'].fillna(embarked_moda)

#
test_df['age'] = test_df['age'].fillna(age_mediana)
test_df['embarked'] = test_df['embarked'].fillna(embarked_moda)

#
y_train = train_df['survived']
y_test = test_df['survived']

#
X_train = pd.get_dummies(
    train_df.drop(columns=['survived']),
    columns=['sex', 'embarked'],
    drop_first=True
)

X_test = pd.get_dummies(
    test_df.drop(columns=['survived']),
    columns=['sex', 'embarked'],
    drop_first=True
)

#
X_test = X_test.reindex(
    columns=X_train.columns,
    fill_value=0
)

print('Train:', X_train.shape)
print('Test :', X_test.shape)
print('X zutabeak:', X_train.columns.tolist())


Train: (712, 6)
Test : (179, 6)
X zutabeak: ['pclass', 'age', 'fare', 'sex_male', 'embarked_Q', 'embarked_S']


## 0.1 Eskalatzea: zer eskalatuko dugu?

KNN distantzietan oinarritzen da. Beraz, eskala oso garrantzitsua da.

Aldagai numerikoak:

```text
age
fare
pclass
```

hauek eskalatuko ditugu.

Dummy aldagaiak:

```text
sex_male
embarked_Q
embarked_S
```

0/1 bezala utziko ditugu.

> KNN-rako eskalatzea ia ezinbestekoa da.  
> Logistic Regression-en **oso gomendagarria** da, baina ez da matematikoki derrigorrezkoa.


In [ ]:
#
num_cols = ['age', 'fare', 'pclass']

#
X_train_s = X_train.copy()
X_test_s = X_test.copy()

#
scaler = StandardScaler()

#
X_train_s[num_cols] = scaler.fit_transform(
    X_train[num_cols]
)

#
X_test_s[num_cols] = scaler.transform(
    X_test[num_cols]
)

print('Prest. Train:', X_train_s.shape[0], '| Test:', X_test_s.shape[0])
display(X_train_s.head())


Prest. Train: 712 | Test: 179


,pclass,age,fare,sex_male,embarked_Q,embarked_S
692,0.829568,-0.081135,0.513812,True,False,True
481,-0.370945,-0.081135,-0.662563,True,False,True
527,-1.571457,-0.081135,3.955399,True,False,True
855,0.829568,-0.887827,-0.467874,False,False,True
801,-0.370945,0.110934,-0.115977,False,False,True


# 1. KNN — «esadazu norekin zabiltzan...» 🟢

KNN-k puntu berri bat sailkatzeko:

1. training adibide guztiekiko distantziak kalkulatzen ditu;
2. hurbilen dauden `k` auzokideak hartzen ditu;
3. gehiengoaren klasea ematen du.

KNN-k ez du Decision Tree bezalako arau multzorik eraikitzen.

> **KNN-ren “ezagutza” training adibideetan dago.**


## 1.1 Distantzia euklidearra — mini-adibidea 🟡

Bi punturen arteko distantzia:

\[
d = \sqrt{\sum (x_i-y_i)^2}
\]

Hutsuneak bete.


In [ ]:
A = np.array([2.5, -1.2])
B = np.array([0.1,  1.6])

# bete ____: distantzia euklidearra kalkulatzeko
distantzia = np.sqrt(((____ - ____) ** _).___())

print('Distantzia:', round(distantzia, 3))


## 1.2 KNN lehen eredua 🟢


In [ ]:
#
knn = KNeighborsClassifier(n_neighbors=5)

#
knn.fit(X_train_s, y_train)

#
print(
    'Accuracy:',
    round(knn.score(X_test_s, y_test), 3)
)

#
y_pred_knn = knn.predict(X_test_s)

#
ConfusionMatrixDisplay(
    confusion_matrix=confusion_matrix(y_test, y_pred_knn),
    display_labels=['Ez bizirik', 'Bizirik']
).plot()

plt.title('KNN — Titanic')
plt.show()


## 1.3 Zergatik eskalatu? 🟢

`fare` bezalako aldagai batek balio askoz handiagoak izan ditzake `age` edo `pclass` baino.

Eskalarik gabe, Euclidean distance kalkuluan `fare`-k gehiegizko pisua har dezake.

Konparatu:


In [ ]:
#
knn_gabe = KNeighborsClassifier(n_neighbors=5)
knn_gabe.fit(X_train, y_train)

#
knn_esk = KNeighborsClassifier(n_neighbors=5)
knn_esk.fit(X_train_s, y_train)

print(
    'Eskalatu gabe:',
    round(knn_gabe.score(X_test, y_test), 3)
)

print(
    'Eskalatuta   :',
    round(knn_esk.score(X_test_s, y_test), 3)
)


# 2. k-ren eragina 🟡

Decision Tree-rekin ikusi genuen:

```text
max_depth handiegia
→ eredu konplexuegia
→ overfitting arriskua
```

KNN-n:

```text
k txikiegia
→ eredu oso lokala
→ zaratarekiko sentikorra
→ overfitting arriskua
```

Eta:

```text
k handiegia
→ auzokide gehiegi
→ erabakia gehiegi leuntzen da
→ underfitting arriskua
```

Hutsuneak bete eta train/test emaitzak gorde.


In [ ]:
k_balioak = [1, 3, 5, 7, 11, 15, 21]

train_scores = []
test_scores = []

for k in k_balioak:

    # bete ____:
    m = KNeighborsClassifier(
        n_neighbors=____
    ).fit(
        ____,
        y_train
    )

    # bete ____:
    train_score = m.score(____, y_train)
    test_score = m.score(____, y_test)

    train_scores.append(train_score)
    test_scores.append(test_score)

    print(
        f'k={k:2d}: '
        f'train={train_score:.3f} '
        f'test={test_score:.3f}'
    )


## 2.1 k-ren grafikoa 🟡

Aurreko Decision Tree-ko `max_depth` grafikoaren antzekoa egingo dugu.

Hutsuneak bete.


In [ ]:
plt.figure(figsize=(9, 5))

# bete ____:
plt.plot(
    ____,
    ____,
    marker='o',
    label='Train'
)

# bete ____:
plt.plot(
    ____,
    ____,
    marker='o',
    label='Test'
)

plt.xlabel('k (auzokide kopurua)')
plt.ylabel('Accuracy')
plt.title('KNN — k balioaren eragina')
plt.legend()
plt.show()


### Hausnartu

1. `k=1`-en train accuracy oso handia da?
2. Test accuracy ere hobetzen da beti `k` txikitzean?
3. Zein `k` ingurutan ikusten duzu train/test oreka ona?
4. Zer antzekotasun dago `k` txikiaren eta Decision Tree-ko `max_depth` handiaren artean?

> Benetako model selection egiteko validation/cross-validation erabiliko dugu gero.  
> Test multzoa azken ebaluaziorako gordetzea da praktika egokiena.


# 3. Erregresio logistikoa — bai/ez-etik probabilitatera 🟢

KNN-k auzokideak erabiltzen ditu.

Erregresio logistikoak beste ikuspegi bat du:

```text
features
   ↓
koefizienteekin puntuazio bat kalkulatu
   ↓
sigmoid
   ↓
P(survived = 1)
   ↓
0.5 atalasea
   ↓
0 edo 1
```

Ez dugu oraindik koefizienteen matematika sakon landuko.

Oraingoz:

> **Logistic Regression = probabilitate bidezko sailkapena**


## 3.1 Zer esan nahi du `max_iter=1000`?

Decision Tree-n `fit()`-ek adarrak eraikitzen zituen modu errekurtsiboan.

Logistic Regression-en `fit()`-ek:

1. hasierako koefiziente batzuk hartzen ditu;
2. errorea neurtzen du;
3. koefizienteak doitzen ditu;
4. berriro errorea neurtzen du;
5. prozesua errepikatzen du konbergitu arte.

```python
max_iter=1000
```

esan nahi du optimizatzaileak **gehienez 1000 iterazio** egiteko aukera duela.

> Ez du esan nahi beti 1000 iterazio egingo dituenik.


In [ ]:
#
lr = LogisticRegression(
    max_iter=1000
)

#
lr.fit(X_train_s, y_train)

#
print(
    'Accuracy:',
    round(lr.score(X_test_s, y_test), 3)
)

#
y_pred_lr = lr.predict(X_test_s)

#
ConfusionMatrixDisplay(
    confusion_matrix=confusion_matrix(y_test, y_pred_lr),
    display_labels=['Ez bizirik', 'Bizirik']
).plot()

plt.title('Logistic Regression — Titanic')
plt.show()


## 3.2 `predict_proba()` — probabilitateak 🟢

`predict()` zuzenean klasea ematen du:

```text
0 edo 1
```

`predict_proba()`-k, aldiz, probabilitateak ematen ditu:

```text
P(0)   P(1)
```

`[:, 1]` erabiliz **1 klasearen probabilitatea** hartzen dugu:

```text
P(bizirik)
```


In [ ]:
#
p = lr.predict_proba(X_test_s)[:, 1]

print(
    'Lehen 5 P(bizirik):',
    np.round(p[:5], 3)
)

#
print(
    '\nLehen 5 probabilitate bikoteak:'
)
print(
    np.round(
        lr.predict_proba(X_test_s)[:5],
        3
    )
)


# 4. Probabilitatea → atalasea → klasea 🟡

Lehenetsitako atalasea:

```text
0.5
```

Beraz:

```text
P(bizirik) >= 0.5 → 1 / Bizirik
P(bizirik) <  0.5 → 0 / Ez bizirik
```

Hutsunea bete:


In [ ]:
atalasea = 0.5
bidaiaria = 4

print(
    'Probabilitatea:',
    round(p[bidaiaria], 3)
)

# bete ____ :
print(
    'Iragarpena:',
    'bizirik'
    if p[bidaiaria] >= ____
    else 'ez'
)


## 4.1 🔴 Erronka — atalasea mugitu

Probatu hurrengo atalase ezberdinak:

```text
0.3
0.5
0.7
```

eta kontatu zenbat bidaiari iragartzen diren **bizirik**.

Hausnartu:

- atalasea jaisten denean, “bizirik” gehiago iragartzen dira?
- atalasea igotzean, zer gertatzen da?

> Geroago ikusiko dugu honek **precision / recall** orekan duen eragina.


In [ ]:
# TODO (🔴): atalaseen konparaketa hemen


# 5. Hiru algoritmoen konparazio mentala

| Algoritmoa | Galdera nagusia | Eskalatzea |
|---|---|---|
| Decision Tree | Zein galderak bereizten ditu klaseak? | Ez da beharrezkoa |
| KNN | Zein training kasu dira hurbilenak? | **Oso garrantzitsua** |
| Logistic Regression | Zein probabilitate du 1 klasekoa izateko? | **Oso gomendagarria** |

Eta `fit()` guztietan agertzen bada ere:

- Decision Tree → adarrak/zatiketak eraikitzen ditu;
- KNN → training datuak prestatzen/gordetzen ditu auzokide-bilaketarako;
- Logistic Regression → koefizienteak iterazioz doitzen ditu.


# Ondorioak

- KNN distantzietan oinarritzen da → **eskala funtsezkoa**.
- `k` txikiegia → overfitting arriskua.
- `k` handiegia → underfitting arriskua.
- Logistic Regression sailkapen algoritmoa da, nahiz eta “regression” izena izan.
- `predict_proba()` → probabilitatea ikusteko.
- 0.5 → sailkapen atalase lehenetsia.
- `max_iter` → Logistic Regression-en optimizazioaren iterazio-muga.
- Preprocessing-ean, train datuekin bakarrik ikasi behar dira mediana, moda eta scaler parametroak → **data leakage saihesteko**.

### Hurrengo pausoa

Hiru ereduak dataset eta split berarekin alderatu:

- Decision Tree
- KNN
- Logistic Regression

eta galdera hau erantzun:

> **Zein modelok orokortzen du hobekien Titanic-en?**
